# 🇮🇳 IndicVoice AI — expressive Indian TTS fine-tuning (Colab T4 16 GB)

Fine-tune a **near-human Indian TTS** voice on expressive Hindi + Gujarati speech from
`ai4bharat/Rasa` and `ai4bharat/IndicVoices`, with checkpoints saved straight to your
Google Drive.

Two tracks are provided:

| Track | Model | VRAM | Why use it |
|---|---|---|---|
| **A (recommended)** | [`ai4bharat/IndicF5`](https://huggingface.co/ai4bharat/IndicF5) — F5-TTS architecture, 11 Indian languages | ~10–14 GB (T4 fits with `batch_size_per_gpu=1600`) | Zero-shot voice cloning from a 5-second reference + emotional prosody. Trained on Rasa/IndicTTS/LIMMITS/IndicVoices-R. |
| **B (optional)** | [`kenpath/svara-tts-v1`](https://huggingface.co/kenpath/svara-tts-v1) — Orpheus-style audio-token LM, Apache-2.0 | ~12 GB in 4-bit | Explicit emotion tags (`<happy>`, `<sad>`, `<anger>`, `<fear>`) and Light LoRA adaptation. |

**Before you start (5 minutes):**

1. `Runtime → Change runtime type → T4 GPU`.
2. Accept the terms on the [IndicF5 model page](https://huggingface.co/ai4bharat/IndicF5) (the repo is gated).
3. Create a Hugging Face **read** token → https://huggingface.co/settings/tokens.
4. In Colab, add it to **Secrets** (key icon, name `HF_TOKEN`).

> The local engine in `src/` keeps using the fast CPU MMS voices from Phases 1–2.
> This notebook produces the *expressive, cloned* voice you can serve from a GPU
> endpoint later.

In [ ]:
!nvidia-smi
%pip -q install -U "transformers>=4.44" datasets soundfile librosa accelerate peft bitsandbytes huggingface_hub

import torch, sys
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Pick Runtime -> Change runtime type -> T4 GPU"

## 1. Authenticate with Hugging Face

In [ ]:
import os
from huggingface_hub import login, whoami

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = ""
if not token:
    from getpass import getpass
    token = getpass("Hugging Face token (must have accepted ai4bharat/IndicF5): ")

login(token=token)
print("signed in as:", whoami()["name"])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

CKPT_DIR = Path("/content/drive/MyDrive/IndicVoiceCheckpoints")
DATA_DIR = Path("/content/data")
for p in (CKPT_DIR, DATA_DIR):
    p.mkdir(parents=True, exist_ok=True)
print("checkpoints ->", CKPT_DIR)
print("data        ->", DATA_DIR)
import shutil; print("free disk    :", round(shutil.disk_usage('/content').free/1e9, 1), "GB")

## 2. Build the expressive Hindi + Gujarati dataset

* **`ai4bharat/Rasa`** — 8 000+ hours of *expressive* Indian speech with emotion labels.
  We keep Hindi + Gujarati and only the four emotions that matter for a conversational
  agent: `happy`, `sad`, `neutral`, `surprise`.
* **`ai4bharat/IndicVoices`** — clean studio read speech, used to keep pronunciation
  stable while the Rasa data adds emotion.

The next two cells are schema-defensive: Hugging Face dataset column names differ
between revisions, so the loader detects them and prints what it found.

In [ ]:
import soundfile as sf, numpy as np, librosa
from datasets import load_dataset

LANG_NAMES = {"hi": "hindi", "gu": "gujarati"}
EMOTIONS = {"happy", "sad", "neutral", "surprise"}
MIN_SEC, MAX_SEC = 1.5, 12.0
MAX_CLIPS_PER_LANG = 3000          # ~6-8 hours of speech; enough for a strong fine-tune
TARGET_SR = 24000                  # F5 / IndicF5 operate at 24 kHz


def pick(columns, *candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def probe(repo, split="train"):
    ds = load_dataset(repo, split=split, streaming=True)
    cols = list(ds.features)
    print(f"{repo} columns:", cols)
    return ds, {
        "text": pick(cols, "text", "transcript", "sentence", "normalized_text", "transcription"),
        "audio": pick(cols, "audio", "wav", "audio_path", "path"),
        "lang": pick(cols, "lang", "language", "language_code", "locale", "lang_code"),
        "emotion": pick(cols, "emotion", "emotion_label", "emotion_name", "label"),
        "speaker": pick(cols, "speaker", "speaker_id", "spk_id"),
    }


rasa, rasa_cols = probe("ai4bharat/Rasa")
print("resolved Rasa columns:", rasa_cols)
assert rasa_cols["text"] and rasa_cols["audio"], "Rasa schema changed - inspect the printed columns"

In [ ]:
def collect(ds, cols, per_lang, require_emotion=False):
    """Write 24 kHz wavs + an F5 metadata.csv (audio_file|text) per language."""
    counts = {lang: 0 for lang in LANG_NAMES}
    for lang in LANG_NAMES:
        (DATA_DIR / lang / "wavs").mkdir(parents=True, exist_ok=True)
        (DATA_DIR / lang / "metadata.csv").write_text("audio_file|text\n", encoding="utf-8")

    for example in ds:
        if all(counts[lang] >= per_lang for lang in LANG_NAMES):
            break
        raw_lang = str(example.get(cols["lang"], "") or "")[:2].lower()
        lang = raw_lang if raw_lang in LANG_NAMES else ""
        if not lang or counts[lang] >= per_lang:
            continue
        emotion = str(example.get(cols["emotion"], "neutral") or "neutral").lower()
        if require_emotion and emotion not in EMOTIONS:
            continue
        text = str(example.get(cols["text"]) or "").strip()
        if not (2 <= len(text) <= 220):
            continue
        audio = example[cols["audio"]]
        wav, sr = np.asarray(audio["array"], dtype=np.float32), int(audio["sampling_rate"])
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if not (MIN_SEC <= len(wav) / sr <= MAX_SEC):
            continue
        if sr != TARGET_SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
        peak = float(np.max(np.abs(wav))) or 1.0
        wav = (wav / peak) * 0.97
        idx = counts[lang]
        path = DATA_DIR / lang / "wavs" / f"{lang}_{idx:05d}.wav"
        sf.write(path, wav, TARGET_SR, subtype="PCM_16")
        with (DATA_DIR / lang / "metadata.csv").open("a", encoding="utf-8") as fh:
            fh.write(f"{path}|{text}\n")
        counts[lang] += 1
        if idx % 500 == 0 and idx:
            print(f"  {lang}: {idx} clips")

    print("clips per language:", counts)
    return counts


counts = collect(rasa, rasa_cols, MAX_CLIPS_PER_LANG, require_emotion=True)

In [ ]:
# Optional: top up with clean studio speech from IndicVoices for stable pronunciation.
try:
    iv, iv_cols = probe("ai4bharat/IndicVoices")
    print("resolved IndicVoices columns:", iv_cols)
    if iv_cols["text"] and iv_cols["audio"]:
        remaining = {lang: 500 for lang in LANG_NAMES}
        collect(iv, iv_cols, 500)
except Exception as exc:
    print("IndicVoices skipped:", type(exc).__name__, exc)

for lang in LANG_NAMES:
    csv = DATA_DIR / lang / "metadata.csv"
    print(lang, "rows:", max(0, len(csv.read_text(encoding='utf-8').splitlines()) - 1))

## 3. Get IndicF5 (F5-TTS architecture)

F5-TTS trains with `accelerate`; datasets are prepared with `prepare_csv_wavs.py`
from a two-column `audio_file|text` CSV (absolute paths).

**Checkpoints on Drive:** the repo writes to `ckpts/`, so we symlink `ckpts/` to
`MyDrive/IndicVoiceCheckpoints/`. A Colab disconnect then cannot lose a run.

In [ ]:
%cd /content
!rm -rf IndicF5
!git clone -q https://github.com/ai4bharat/IndicF5.git
%cd /content/IndicF5
!pip -q install -r requirements.txt
!pip -q install -U accelerate soundfile librosa

import torch, transformers
print("transformers:", transformers.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

# Gated repo: accept the terms once on the model page, then this download works.
INDICF5_DIR = Path(snapshot_download(
    "ai4bharat/IndicF5",
    local_dir=str(CKPT_DIR / "IndicF5"),
    allow_patterns=["*.json", "*.safetensors", "*.pt", "*.txt", "*.yaml", "*.md"],
))
print("IndicF5 weights:", INDICF5_DIR)
print(sorted(p.name for p in INDICF5_DIR.iterdir())[:20])

In [ ]:
# Build F5 training data: data/<lang>_char/{raw.pt, duration.json, vocab.txt}
import subprocess, sys

for lang in ("hi", "gu"):
    csv_path = DATA_DIR / lang / "metadata.csv"
    print(f"== preparing {lang} from {csv_path}")
    result = subprocess.run(
        [sys.executable, "src/f5_tts/train/datasets/prepare_csv_wavs.py",
         str(csv_path), str(DATA_DIR)],
        capture_output=True, text=True,
    )
    print(result.stdout[-1500:] or result.stderr[-1500:])

print(sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

## 4. Fine-tune on the T4

Key choices for a **16 GB T4**:

* `--batch_size_type frame --batch_size_per_gpu 1600` — ~17 s of audio per step, fits comfortably.
* `--grad_accumulation_steps 4` — effective batch ≈ 68 s of audio.
* `--epochs 2` first; resume from the last checkpoint for more.
* `--save_per_updates 200 --keep_last_n_checkpoints 2` — every ~10 min, and Drive stays small.
* `--finetune True --pretrain <IndicF5 checkpoint>` — start from the 11-language Indic model.

Runs ~1.5–2.5 h on a free T4 for the data above. Watch the loss in the log; it should
fall steadily instead of bouncing.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/IndicF5")
ckpt_link = REPO / "ckpts"
if ckpt_link.is_symlink():
    pass
elif ckpt_link.exists():
    print("existing ckpts/ kept:", sorted(p.name for p in ckpt_link.iterdir())[:5])
else:
    ckpt_link.symlink_to(CKPT_DIR, target_is_directory=True)
print("ckpts ->", ckpt_link.resolve())

# Pick the IndicF5 checkpoint file shipped with the weights.
pretrain = None
for pattern in ("*.safetensors", "*.pt"):
    found = sorted((CKPT_DIR / "IndicF5").glob(pattern))
    if found:
        pretrain = found[0]
        break
print("pretrain:", pretrain)
assert pretrain, "No IndicF5 checkpoint found - re-run the download cell"

DATASET_NAME = "hi_char"   # produced by prepare_csv_wavs.py; switch to gu_char for Gujarati

In [ ]:
import subprocess, os

cmd = [
    "accelerate", "launch",
    "src/f5_tts/train/finetune_cli.py",
    "--exp_name", "F5TTS_v1_Base",
    "--dataset_name", DATASET_NAME,
    "--finetune", "True",
    "--pretrain", str(pretrain),
    "--tokenizer", "char",
    "--learning_rate", "1e-5",
    "--batch_size_per_gpu", "1600",
    "--batch_size_type", "frame",
    "--max_samples", "32",
    "--grad_accumulation_steps", "4",
    "--max_grad_norm", "1.0",
    "--epochs", "2",
    "--num_warmup_updates", "100",
    "--save_per_updates", "200",
    "--keep_last_n_checkpoints", "2",
    "--last_per_updates", "400",
    "--logger", "tensorboard",
]
print(" ".join(cmd))
env = dict(os.environ, PYTHONUNBUFFERED="1", ACCELERATE_MIXED_PRECISION="fp16")
subprocess.run(cmd, cwd="/content/IndicF5", env=env, check=False)
print("checkpoints on Drive:", sorted(p.name for p in (CKPT_DIR / DATASET_NAME).glob("*"))[-5:] if (CKPT_DIR / DATASET_NAME).exists() else "none yet")

### If you hit CUDA out-of-memory

Lower `--batch_size_per_gpu` to `1000`, then `--max_samples` to `16`. If it still OOMs,
use the LoRA route below (an adapter of ~12 MB keeps the frozen base in fp16 and only
trains low-rank matrices).

```bash
accelerate launch src/f5_tts/train/finetune_cli.py \
  --exp_name F5TTS_v1_Base --dataset_name hi_char --finetune --lora \
  --lora_rank 16 --lora_alpha 16 --learning_rate 1e-4 --epochs 1 \
  --batch_size_per_gpu 3200 --batch_size_type frame --tokenizer char
```

The `--lora` flag exists in F5-TTS forks such as
[`instavar/f5-tts-lora-finetuning`](https://github.com/instavar/f5-tts-lora-finetuning);
check the fork's flags before running, and keep `--tokenizer char` for Indic scripts.

## 5. Inference: 5-second zero-shot cloning + emotional prosody

Give it a 5–10 s reference clip of the voice you want, its transcript, and the text to
say. Emotion in Indian TTS comes mostly from the reference clip's delivery plus
punctuation — exclamation marks, short sentences and natural fillers lift prosody
noticeably compared with flat read speech.

In [ ]:
from IPython.display import Audio, display
from pathlib import Path
import numpy as np

# Use a clip from your own dataset as the voice reference (5-10 s works best).
REF_AUDIO = str(next((DATA_DIR / "hi" / "wavs").glob("*.wav"), ""))
REF_TEXT = (DATA_DIR / "hi" / "metadata.csv").read_text(encoding="utf-8").splitlines()[1].split("|", 1)[1]
print("reference:", REF_AUDIO, "|", REF_TEXT[:80])

CHECKPOINT = sorted((CKPT_DIR / DATASET_NAME).glob("model_*.safetensors"))[-1] if (CKPT_DIR / DATASET_NAME).exists() else pretrain
print("using checkpoint:", CHECKPOINT)

from transformers import AutoModel
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True).to("cuda").eval()

def speak(text, out_path):
    result = model(text, ref_audio_path=REF_AUDIO, ref_text=REF_TEXT)
    wav, sr = result if isinstance(result, tuple) else (result, 24000)
    wav = np.asarray(wav, dtype=np.float32)
    import soundfile as sf
    sf.write(out_path, wav, sr)
    print("wrote", out_path, f"{len(wav)/sr:.2f}s")
    display(Audio(wav, rate=sr))

# Neutral vs emotional delivery of the same sentence
speak("નમસ્તે, હું તમારી મદદ કરી શકું છું. કહો, શું જોઈએ છે?", "/content/gu_neutral.wav")
speak("વાહ! આ તો ખૂબ સરસ છે! મને ખૂબ આનંદ થયો!", "/content/gu_happy.wav")
speak("नमस्ते, मैं आपकी मदद कर सकता हूँ। बताइए, क्या चाहिए?", "/content/hi_neutral.wav")
speak("अरे नहीं! यह तो बहुत बुरी खबर है। मुझे बहुत दुख हुआ।", "/content/hi_sad.wav")

for a, b in (("/content/hi_neutral.wav", "/content/hi_sad.wav"), ("/content/gu_neutral.wav", "/content/gu_happy.wav")):
    print(a, "vs", b, "-> listen for the pitch/energy difference")

## 6. Optional: svara-tts-v1 (explicit emotion tags + LoRA)

`kenpath/svara-tts-v1` is an Orpheus-style audio-token model with **19 Indian
languages** (Hindi, Gujarati, …) and emotion tags, Apache-2.0. It is a causal-LM over
discrete audio tokens instead of a flow-matching model, so training looks like an LLM
fine-tune: tokenize audio with their codec, then LoRA the transformer.

Follow the tokenizer/preparation steps from the model card / their Colab first, then
this is the LoRA wiring:

In [ ]:
# Template - run only after tokenizing your audio with svara's codec (see model card).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "kenpath/svara-tts-v1"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
base = prepare_model_for_kbit_training(base)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()

print("Train with TRL SFTTrainer on tokenized (text + audio-token) rows, then save:")
print("  model.save_pretrained('/content/drive/MyDrive/IndicVoiceCheckpoints/svara-lora')")
print("Prompt style for emotion: '<happy> નમસ્તે, આજે તમે કેમ છો?'")

## 7. Where this fits your laptop engine

| Piece | Runs on | File |
|---|---|---|
| Mic → VAD → STT → LLM → streaming TTS | your i7 CPU | `src/sts_pipeline.py` |
| Fast Hindi/Gujarati/English voices (MMS, ONNX) | your i7 CPU | `src/tts_engine.py` |
| Expressive cloned voice from this notebook | Colab GPU / any GPU endpoint | exported F5 checkpoint |

Once the fine-tune sounds right, serve it from a GPU box (or a Colab notebook with a
FastAPI endpoint) and point a future `tts_engine.py` backend at it — the local MMS voice
stays the always-available fallback.

## 8. Teacher → Student distillation for the laptop CPU

The voices in `models/tts-*/` are Meta **MMS VITS** checkpoints (~110 MB each):
they run on your i7 at RTF 0.4–0.7 and load in sherpa-onnx with no GPU. The
expressive teacher from this notebook (**IndicF5**) needs a GPU at inference, so
the goal here is to move its *expressiveness* into that fast CPU voice.

Pipeline:

1. **Teacher** (this Colab, T4): generate a multi-emotion corpus with IndicF5.
2. **Student** (this Colab, T4): fine-tune a VITS/MMS student on the teacher audio.
3. **Export** → INT8 ONNX → copy into `models/tts-<lang>/model.onnx` on the laptop.

> Honest status: steps 1 and 3 below are complete, runnable code. Step 2 needs an
> external VITS trainer — the Hugging Face recipe
> [`ylacombe/finetune-hf-vits`](https://github.com/ylacombe/finetune-hf-vits) is
> the maintained one. The cell prints the command to run instead of pretending to
> know its current flags; check that repo's README when you get there.

In [ ]:
# ---- Step 1: teacher corpus, four emotions x hi/gu -------------------------
EMOTIONS = {
    "happy":     "वाह! यह तो बहुत अच्छी खबर है!",
    "empathetic": "मुझे समझ आ रहा है, आप परेशान मत होइए।",
    "calm":      "ठीक है, धीरे-धीरे करते हैं। कोई जल्दी नहीं है।",
    "urgent":    "सुनिए, यह काम आज ही होना ज़रूरी है!",
}
GU = {
    "happy":     "વાહ! આ તો ખૂબ સરસ સમાચાર છે!",
    "empathetic": "મને સમજાય છે, તમે ચિંતા ન કરો.",
    "calm":      "ઠીક છે, ધીરે ધીરે કરીએ. કોઈ ઉતાવળ નથી.",
    "urgent":    "સાંભળો, આ કામ આજે જ કરવું પડશે!",
}

from pathlib import Path
import soundfile as sf

DISTILL = Path("/content/distill")
for lang, bank in (("hi", EMOTIONS), ("gu", GU)):
    wav_dir = DISTILL / lang / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for emotion, sentence in bank.items():
        out = wav_dir / f"{lang}_{emotion}.wav"
        if not out.exists():
            result = model(sentence, ref_audio_path=REF_AUDIO, ref_text=REF_TEXT)
            wav, sr = result if isinstance(result, tuple) else (result, 24000)
            sf.write(out, wav, sr)
        rows.append(f"{out}|{sentence}")
        print(emotion, "->", out.name, "ok")
    (DISTILL / lang / "metadata.csv").write_text(
        "audio_file|text\n" + "\n".join(rows) + "\n", encoding="utf-8"
    )
print("teacher corpus ready in", DISTILL)
print("Scale up by looping your Rasa sentences through the same model() call.")

### Step 2: train the student

Two viable students — pick one:

| Student | Pros | Cons |
|---|---|---|
| **VITS / MMS** (`facebook/mms-tts-hin`, `…-guj`) | exactly the architecture already in `models/`, sherpa-onnx native, CPU RTF ≈ 0.5 | needs the adversarial VITS trainer; expressiveness ceiling is lower than F5 |
| **Matcha-TTS** | faster than VITS, ONNX-exportable to sherpa-onnx | needs its own trainer + a separate vocoder |

For VITS, clone the HF recipe and run its script against the corpus above
(`audio_file|text` CSV in `metadata.csv` is the same format it expects):

```bash
git clone https://github.com/ylacombe/finetune-hf-vits && cd finetune-hf-vits
pip install -r requirements.txt
# read its README for the current arguments, then point it at:
#   model      = facebook/mms-tts-hin   (or -guj)
#   dataset    = /content/distill/hi    (metadata.csv + wavs/)
#   output_dir = /content/drive/MyDrive/IndicVoiceCheckpoints/vits-student-hi
accelerate launch run_vits_finetuning.py ...   # see repo README for flags
```

### Step 3: export the student to INT8 ONNX for the laptop

Convert the fine-tuned checkpoint to ONNX with the same exporter the MMS models
were built with — [`willwade/mms-tts-multilingual-models-onnx`](https://github.com/willwade/mms-tts-multilingual-models-onnx)
(`vits-mms.py`, produces `model.onnx` + `tokens.txt`) — then quantize here:

In [ ]:
# Dynamic INT8 quantization of an exported student model.
# `pip install onnx onnxruntime` on Colab, then set STUDENT_ONNX to the exported file.
from pathlib import Path

STUDENT_ONNX = Path("/content/drive/MyDrive/IndicVoiceCheckpoints/student-hi/model.onnx")
OUT_ONNX = STUDENT_ONNX.with_name("model.int8.onnx")

if STUDENT_ONNX.exists():
    from onnxruntime.quantization import QuantType, quantize_dynamic

    quantize_dynamic(str(STUDENT_ONNX), str(OUT_ONNX), weight_type=QuantType.QInt8)
    print("quantized ->", OUT_ONNX, round(OUT_ONNX.stat().st_size / 1e6, 1), "MB")
    print()
    print("Copy both files to the laptop, keeping the names sherpa expects:")
    print("  models/tts-hi/model.onnx   <- quantized student")
    print("  models/tts-hi/tokens.txt   <- from the exporter")
    print()
    print("Then verify on the laptop:")
    print('  python -m src.sts_pipeline --say "नमस्ते! यह नई आवाज़ है।"')
else:
    print("Export the fine-tuned student to ONNX first (see the willwade exporter), then re-run this cell.")


### Checking it on the laptop

`sherpa-onnx` expects a **VITS** graph with the same input/output signature as the
MMS models. After copying, `TTSEngine` loads it unchanged:

```powershell
python -m src.sts_pipeline --say "નમસ્તે! આ નવો અવાજ છે." --lang gu
```

If loading fails, compare the exported graph's inputs/outputs against the current
working model (`models/tts-gu/model.onnx`) — the exporter version matters more
than the checkpoint.